# 07 — Évaluation en boucle fermée (Phase 9)

Visualise l'impact **causal** des contrôleurs mesuré en pilotant réellement
jumeaux-chauds (`evaluation/closed_loop_eval.py`). Contrairement au benchmark
offline (notebook 05), ici la physique du simulateur recalcule les températures
en réponse aux RPM commandés — `T_mean`, l'énergie et le PUE diffèrent donc
réellement d'un contrôleur à l'autre.

**Prérequis :** un fichier de résultats produit par
`python -m evaluation.closed_loop_eval --scenario stress ...`

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

SCENARIO = "stress"
_candidates = [
    Path(f"../evaluation/results/closed_loop_results_{SCENARIO}.json"),
    Path(f"evaluation/results/closed_loop_results_{SCENARIO}.json"),
]
path = next((p for p in _candidates if p.exists()), None)
assert path is not None, (
    "Résultats absents. Lance d'abord :\n"
    "  python -m evaluation.closed_loop_eval --scenario stress --duration 300 --dt 5 --speed 60 \\\n"
    "    --controllers native supervised score_controller baseline_pid baseline_fixed_4500"
)
payload = json.loads(path.read_text())
print(f"Source : {path}")
print(f"scénario={payload['scenario']}  durée={payload['duration_s']}s  dt={payload['dt_s']}s  vitesse={payload['speed']}x")

In [ ]:
df = pd.DataFrame(payload["results"])
cols = ["name", "nb_shutdowns_cl", "nb_avoidable", "nb_inevitable",
        "nb_avoidable_avoided", "T_mean_cl", "T_max_cl", "rpm_mean_cl",
        "pue_mean", "energy_fans_kwh", "energy_saved_vs_max_pct"]
df = df[[c for c in cols if c in df.columns]].set_index("name")
df.round(3)

## 1. Front de Pareto sécurité ↔ sobriété

Chaque contrôleur est un compromis : **plus froid** (sécurité, axe Y vers le bas)
coûte généralement **plus d'énergie** (axe X vers la droite). Le coin
bas-gauche est l'idéal (froid *et* sobre).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["energy_fans_kwh"], df["T_mean_cl"], s=90, color="tab:blue", zorder=3)
for name, row in df.iterrows():
    ax.annotate(name, (row["energy_fans_kwh"], row["T_mean_cl"]),
                xytext=(6, 4), textcoords="offset points", fontsize=9)
ax.set_xlabel("Énergie fans (kWh) — sobriété →")
ax.set_ylabel("Température moyenne (°C) — sécurité ↓")
ax.set_title(f"Pareto sécurité/énergie — scénario {payload['scenario']}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Énergie fans, PUE et RPM par contrôleur

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["energy_fans_kwh"].plot.bar(ax=axes[0], color="tab:orange", title="Énergie fans (kWh)")
df["pue_mean"].plot.bar(ax=axes[1], color="tab:green", title="PUE moyen")
df["rpm_mean_cl"].plot.bar(ax=axes[2], color="tab:purple", title="RPM moyen commandé")
for ax in axes:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
axes[1].axhline(1.0, color="grey", ls="--", lw=0.8)
plt.tight_layout()
plt.show()

## 3. Pannes : évitables vs inévitables

Une panne est **inévitable** si une `fan_failure` est active (RPM forcé à 0
quelle que soit la commande). Les autres sont **évitables** par un meilleur
contrôle. `nb_avoidable_avoided` compte les pannes évitables épargnées vs le
contrôleur natif.

> Note : sur une fenêtre courte de scénario `stress`, les températures peuvent
> rester sous le seuil de shutdown — dans ce cas tous les compteurs valent 0 et
> la valeur ajoutée se lit surtout sur l'énergie et le PUE. La mécanique de
> classification des pannes est validée par `tests/test_phase9_closed_loop.py`.

In [ ]:
shut = df[["nb_avoidable", "nb_inevitable"]]
if shut.to_numpy().sum() == 0:
    print("Aucun shutdown sur cette fenêtre — comparer plutôt l'énergie/PUE ci-dessus.")
else:
    ax = shut.plot.bar(stacked=True, figsize=(8, 4),
                       color=["tab:red", "tab:grey"])
    ax.set_ylabel("Nombre de shutdowns")
    ax.set_title("Pannes évitables (rouge) vs inévitables (gris)")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()

## Synthèse

- La boucle fermée **différencie causalement** les contrôleurs, ce que
  l'évaluation offline ne pouvait pas faire (mêmes données rejouées).
- Lecture type (scénario `stress`) : `baseline_fixed_4500` est le plus froid
  mais de loin le plus énergivore ; `supervised` refroidit utilement pour une
  fraction de l'énergie ; un contrôleur trop sobre (`score_controller`) laisse
  `T_max` approcher dangereusement le seuil.
- Voir `documents/rapport_analyse.md` (section 8) pour l'analyse détaillée.